# Layer C — theme summarization, tested on the real corpus

Connects to the **real Elastic + Neo4j** (`config.example.yaml`), then:

1. **§2** sanity-checks that Layer B themes exist.
2. **§3** runs the **read-only benchmark** (`benchmark_theme`) over the largest themes — all 5 extractive
   methods (LexRank / LSA / Luhn / TextRank / NLTK-freq), side by side, with the per-method JSON metrics
   (`input_chunks`, `input_chars`, `execution_time_ms`, `summary_length_chars`). **No LLM, writes nothing.**
3. **§4** runs the full pipeline on **ONE theme** (extractive Stage 1 → LLM Stage 2 → `Card`) so you can
   see the final card. One LLM call, persists nothing — it does **not** run across the whole facet.

> Prereqs: `docker compose up -d` (Neo4j + Elastic), Layer A + B already built in them, and env vars
> `NEO4J_PASSWORD` (+ `GROQ_API_KEY` for §4). Run in the `foodscholar` conda env.

## 1 · Connect to the real stores

In [ ]:
import os
from pathlib import Path

# repo root (works whether the kernel starts in notebooks/ or the repo root)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
CONFIG = ROOT / "config.example.yaml"
assert CONFIG.exists(), f"missing {CONFIG}"
os.environ["NEO4J_PASSWORD"] = "password"
os.environ.setdefault("GROQ_API_KEY", "***REMOVED-GROQ-KEY***")
# Heads-up if the secrets the config interpolates aren't set.
for var in ["NEO4J_PASSWORD"]:
    if not os.environ.get(var):
        print(f"⚠ {var} not set — Neo4j connect will fail (export it or edit the config).")

from foodscholar import FoodScholar

fs = FoodScholar.from_config(CONFIG)
print(fs.info())

{'foodscholar': '0.1.0', 'config_hash': '04975469844a1798', 'chunk_store': 'elastic', 'graph_store': 'neo4j', 'embedder': 'lazy(BAAI/bge-base-en-v1.5)', 'llm': 'fallback(llama-3.3-70b-versatile,llama3.1)', 'ontology': 'configured', 'ner': 'gliner', 'nel_backend': 'hnsw', 'prompt_version': 'v1'}


## 2 · Sanity-check Layer B themes exist
Layer C summarizes **Layer B themes**. If this is empty, build Layer A+B first (`fs.build_layer_b()` or your pipeline).

In [4]:
FACET = "foods"

themes = [t for t in fs.graph.themes() if t.model.facet == FACET]
themes_sorted = sorted(themes, key=lambda t: t.chunk_count, reverse=True)
print(f"{len(themes_sorted)} themes in facet '{FACET}'")
for t in themes_sorted[:10]:
    print(f"  {t.chunk_count:>5} chunks  {t.theme_id:24} {t.label[:50]}")
assert themes_sorted, "no Layer B themes — build Layer A+B before testing Layer C"


ServiceUnavailable: Couldn't connect to localhost:7687 (resolved to ('127.0.0.1:7687',)):
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [Errno 111] Connection refused)

## 3 · Read-only benchmark — the 5 extractive methods, side by side

`fs.benchmark_layer_c(...)` runs every method over the largest `themes` and writes a combined JSON to
`config.layer_c.benchmark_out_dir`. Below we also render the summaries inline so you can judge coverage /
redundancy / coherence by eye. **No LLM, persists nothing.**

In [ ]:
# how many sentences each extractive method keeps; bump for longer extracts
fs.config.layer_c.stage1_sentences = 8

N_THEMES = 5
results = fs.benchmark_layer_c(facet=FACET, themes=N_THEMES)   # {theme_id: [MethodResult, ...]}

import pandas as pd
rows = []
for tid, methods in results.items():
    for r in methods:
        rows.append({"theme": tid, "method": r.method, "in_chunks": r.input_chunks,
                     "in_chars": r.input_chars, "ms": r.execution_time_ms,
                     "out_chars": r.summary_length_chars})
df = pd.DataFrame(rows)
display(df.sort_values(["theme", "ms"]).reset_index(drop=True))
print("combined JSON written under:", fs.config.layer_c.benchmark_out_dir)

### 3b · Eyeball the actual summaries for one theme
The whole question of the baseline: *are these good building blocks for the LLM stage?*

In [ ]:
THEME_ID = themes_sorted[0].theme_id   # largest theme; set to any id from §2

print(f"theme: {THEME_ID}  —  {fs.graph.theme(THEME_ID).label}\n")
for r in results[THEME_ID]:
    print("=" * 80)
    print(f"[{r.method}]  {r.execution_time_ms} ms · {r.input_chunks} chunks "
          f"· {r.input_chars} chars in → {r.summary_length_chars} chars out")
    print("-" * 80)
    print(r.summary)
    print()

## 4 · Full pipeline on ONE theme — extractive → LLM → Card

Adds **Stage 2** (LLM refinement) on top of the configured Stage-1 method for a **single theme**
(`THEME_ID` from §3) — **one LLM call**, not the whole facet. Needs `GROQ_API_KEY`; persists nothing.

> This deliberately does NOT call `fs.build_layer_c(facet=...)`, which would summarize *every* theme
> (hundreds of LLM calls). For a real full run, use `fs.build_layer_c(facet=FACET)` outside this notebook.

In [ ]:
from foodscholar.layer_c.registry import build_summarizer
from foodscholar.layer_c.stage1 import run_stage1
from foodscholar.layer_c.stage2 import run_stage2
from foodscholar.layer_c.builder import _ThemeAdapter

if not os.environ.get("GROQ_API_KEY"):
    print("⚠ GROQ_API_KEY not set — Stage 2 will fail. Set it to run this cell.")

# pick the Stage-1 method you liked best in §3
fs.config.layer_c.stage1_method = "lexrank"   # lexrank | lsa | luhn | textrank | nltk_freq
cfg = fs.config.layer_c

th = fs.graph.theme(THEME_ID)
chunk_ids = list(fs.graph_store.get_chunks_for_theme(THEME_ID))
texts = [c.text for c in fs.chunk_store.get_many(chunk_ids)]

summarizer = build_summarizer(cfg.stage1_method, cfg)
s1 = run_stage1(texts, summarizer,
                map_reduce_threshold=cfg.map_reduce_threshold,
                group_char_budget=cfg.group_char_budget)
print(f"theme: {THEME_ID}  —  {th.label}")
print(f"Stage 1 ({cfg.stage1_method}, {s1.strategy}, {s1.n_groups} group(s)): "
      f"{s1.n_input_chunks} chunks → {len(s1.text)} chars\n")
print(s1.text, "\n")

adapter = _ThemeAdapter(theme_id=th.theme_id, label=th.label,
                        facet=th.model.facet, keyword_terms=list(th.model.keyword_terms))
card = run_stage2(fs.llm, s1, adapter, chunk_ids, cfg)   # ONE LLM call

print("=" * 80, "\nCARD")
print("title:           ", card.title)
print("evidence_quality:", card.evidence_quality)
print("tip:             ", card.tip)
print("safety_flagged:  ", card.safety_flagged)
print("cited chunks:    ", len(card.cited_chunk_ids))
print("\nsummary:\n", card.summary)

## 5 · Embed the card + vector-search it

Layer C cards now carry an embedding (`title + summary`, via `fs.embedder`) and are stored in the
Elastic `foodscholar_cards` index. The full `fs.build_layer_c()` does this for every card; here we embed
just the §4 card, upsert it, and run `fs.search_cards(...)` to confirm it's retrievable.

> `fs.search_cards(text, k=...)` embeds the query, kNN's the card index, and returns the nearest cards.
> (Thin retrieval helper — the full `query()` with answer synthesis is still deferred.)

In [ ]:
# embed the single §4 card and store it (build_layer_c does this for all cards)
vec = fs.embedder.embed([f"{card.title}\n\n{card.summary}"])[0]
card.embedding = vec
card.embedding_model = fs.embedder.model_id
fs.card_store.init()
fs.card_store.upsert([card])
print(f"stored card {card.card_id} with {len(vec)}-dim embedding ({card.embedding_model})")

# vector-search the cards by a natural-language query
for q in ["calcium and vitamin D for bones", "lactose intolerance dairy alternatives"]:
    hits = fs.search_cards(q, k=3)
    print(f"\nquery: {q!r} → {len(hits)} card(s)")
    for c in hits:
        print(f"  [{c.target_id}] {c.title}")